# Adversarial Patch Experiments — YOLOv8 / YOLO11 / YOLO26

**Capstone: Person-Vanishing Adversarial Patch Attack against Ultralytics YOLO**

This notebook trains and evaluates adversarial patches against YOLOv8n, YOLO11n,
and YOLO26n, then runs cross-version transfer tests.

**Before running:** `Runtime > Change runtime type > T4 GPU`

## 1. Setup

In [ ]:
# Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Clone the repo
REPO_URL = 'https://github.com/Cmaddock99/Patch_Yolo.git'
REPO_DIR = '/content/Adversarial_Patch'
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python-headless tqdm -q

## 2. Train Patch — YOLOv8n (1000 epochs)

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolov8n \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
from IPython.display import Image as IPImage, display
import json, os
display(IPImage('outputs/yolov8n_patch_v1/patches/patch.png', width=200))
with open('outputs/yolov8n_patch_v1/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
from IPython.display import Image as IPImage, display
import os
for orig, patched in zip(
    sorted(os.listdir('outputs/yolov8n_patch_v1/original'))[:3],
    sorted(os.listdir('outputs/yolov8n_patch_v1/patched'))[:3]):
    print('CLEAN:', orig)
    display(IPImage(f'outputs/yolov8n_patch_v1/original/{orig}', width=400))
    print('PATCHED:', patched)
    display(IPImage(f'outputs/yolov8n_patch_v1/patched/{patched}', width=400))

## 3. Train Patch — YOLO11n (1000 epochs)

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolo11n \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
import json
with open('outputs/yolo11n_patch_v1/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 4. Train Patch — YOLO26n (1000 epochs)

YOLO26 is NMS-free. Run the check cell first to confirm the model name.

In [ ]:
# Confirm YOLO26 model name in current Ultralytics version
import ultralytics
print('Ultralytics version:', ultralytics.__version__)
from ultralytics import YOLO
try:
    m = YOLO('yolo26n.pt')
    print('yolo26n loaded OK')
except Exception as e:
    print('Error:', e)
    print('Check https://docs.ultralytics.com/models/yolo26/ for the correct name')

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolo26n \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

## 5. Cross-Version Transfer Tests

Core research question: does a patch trained on YOLOv8 transfer to YOLO11/v26?
This extends Zimon (2025) to YOLO26 — the primary capstone contribution.

In [ ]:
# v8 patch → evaluated on v11
!python experiments/ultralytics_patch.py \
    --model yolo11n --eval-only \
    --load-patch outputs/yolov8n_patch_v1/patches/patch.png \
    --display 3

In [ ]:
# v8 patch → evaluated on v26
!python experiments/ultralytics_patch.py \
    --model yolo26n --eval-only \
    --load-patch outputs/yolov8n_patch_v1/patches/patch.png \
    --display 3

In [ ]:
# v11 patch → evaluated on v26
!python experiments/ultralytics_patch.py \
    --model yolo26n --eval-only \
    --load-patch outputs/yolo11n_patch_v1/patches/patch.png \
    --display 3

## 6. Results Summary Table

In [ ]:
import json, os
print(f"{'Experiment':<40} {'Clean':>6} {'Patched':>8} {'Suppression':>12}")
print('-' * 70)
for run_dir in sorted(os.listdir('outputs')):
    rp = f'outputs/{run_dir}/results.json'
    if os.path.exists(rp):
        with open(rp) as f:
            r = json.load(f)
        print(f"{run_dir:<40} {r.get('clean_person_detections','-'):>6} "
              f"{r.get('patched_person_detections','-'):>8} "
              f"{str(r.get('detection_suppression_pct','-'))+'%':>12}")

## 7. Push Results to GitHub

In [ ]:
!git config user.email 'your@email.com'
!git config user.name 'Cmaddock99'
!git add outputs/ experiments/
!git commit -m 'Add Colab experiment results: yolov8n / yolo11n / yolo26n'
# Paste a GitHub Personal Access Token with repo write access below
# Create one at: GitHub > Settings > Developer settings > Personal access tokens
TOKEN = 'YOUR_GITHUB_PAT_HERE'
REPO  = 'Cmaddock99/Patch_Yolo'
import subprocess
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{TOKEN}@github.com/{REPO}.git'])
!git push